In [ ]:
# Importerer alle nødvendige biblioteker
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

DATA_PATH = Path("../../../outputs/outputs/all_matches_features_final.csv")


Imports OK ✓


In [ ]:
# 12 features — bedste konfiguration fra ablation studie
FEATURES = [
    "distance_m",
    "angle_rad",
    "pressure_nd_dist_m",
    "pressure_def_count_r2m",
    "shooter_speed_mps",
    "ball_speed_mps",
    "shot_body_part",
    "play_pattern",
    "gk_ball_distance",
    "obstruction_count",
    "gk_lateral_offset",
    "gk_depth",
]
print(f"Antal features: {len(FEATURES)}")

In [3]:
# Vi åbner vor datafil med alle 5056 skud fra alle 192 kampe i Superliga 2024/25.


df = pd.read_csv(DATA_PATH)
df["is_goal"] = pd.to_numeric(df["is_goal"], errors="coerce")
df = df.loc[df["is_goal"].isin([0, 1])].reset_index(drop=True)

print(f"Totalt antal skud : {len(df)}")
print(f"Mål               : {int(df['is_goal'].sum())} ({df['is_goal'].mean()*100:.1f}%)")
print(f"Kampe             : {df['game_id'].nunique()}")

Totalt antal skud : 5056
Mål               : 601 (11.9%)
Kampe             : 192


In [4]:
# Data til modellen, vi konverter fx header, right foot til tal, hvorefter vi deller op i 80/20 til træning og test dertil manglende værdier fyldes ud med gennemsnit

feat_names = [f for f in FEATURES if f in df.columns]

def encode(df_sub):
    X = df_sub.copy()
    for col in X.columns:
        if X[col].dtype == object or str(X[col].dtype) == "category":
            codes, _ = pd.factorize(X[col])
            X[col] = codes.astype(float)
        else:
            X[col] = pd.to_numeric(X[col], errors="coerce")
    return X.to_numpy(dtype=float)

X_raw = encode(df[feat_names])
y     = df["is_goal"].astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)

medians = np.nanmedian(X_train, axis=0)
for j in range(X_train.shape[1]):
    X_train[np.isnan(X_train[:, j]), j] = medians[j]
    X_test[np.isnan(X_test[:, j]), j]   = medians[j]

print(f"Træning : {X_train.shape[0]} skud")
print(f"Test    : {X_test.shape[0]} skud")

Træning : 4044 skud
Test    : 1012 skud


In [5]:
# og så træner vi vores TabPFN model på træningsættet

from tabpfn import TabPFNClassifier

tabpfn = TabPFNClassifier(device="cpu", fit_mode="fit_with_cache", ignore_pretraining_limits=True)
tabpfn.fit(X_train, y_train)

y_prob = tabpfn.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
ll  = log_loss(y_test, y_prob)
print(f"TabPFN — AUC: {auc:.4f} | Log-loss: {ll:.4f}")

TabPFN — AUC: 0.8961 | Log-loss: 0.2060


In [6]:
# hertil giver hver skud en xg-værdi altså alle 5056 skud = som ender med xg-værdi 

# yderlig så finder vi gennemsnittet af mål, som skal passe med det vi ved fra datasættet som er 11 proccent

X_full = encode(df[feat_names])
for j in range(X_full.shape[1]):
    X_full[np.isnan(X_full[:, j]), j] = medians[j]

df["xg"] = tabpfn.predict_proba(X_full)[:, 1]
print(f"Gennemsnitlig xG: {df['xg'].mean():.4f}")

Gennemsnitlig xG: 0.1098


In [7]:
# tag e kamp game_ID og filterer data (skuddene derfra) tilfædig

df["game_id"].unique()[9]

# filtrere alle skudene ud her, og kamp:df har alle skudne nu fra kampen
kamp_df = df[df["game_id"] == 2442554]



In [ ]:
# Simulation for hele kampen — 1000 iterationer
# Hvert skud simuleres som Bernoulli(p_i): random tal < xG = mål
simulated_goals_match = []

for i in range(1000):
    random_numbers = np.random.rand(len(kamp_df))
    goals = random_numbers < kamp_df["xg"].values
    total_goals = goals.sum()
    simulated_goals_match.append(total_goals)

print(f"Gennemsnitlige simulerede mål i kampen: {np.mean(simulated_goals_match):.2f}")

In [ ]:

# se kampen
kamp_df["is_goal"]

229    0
230    0
231    0
232    0
233    0
234    0
235    0
236    1
237    0
238    0
239    0
240    0
241    0
242    0
243    0
244    0
245    0
246    0
247    0
248    0
249    1
250    0
251    0
252    0
253    0
254    0
255    0
256    0
257    1
Name: is_goal, dtype: int64

In [ ]:
# split holdene op hertil 

silkeborg_df = kamp_df[kamp_df["team_id"] == 418]


aab_df = kamp_df[kamp_df["team_id"] == 401]



In [ ]:
# Simulation for Silkeborg — 1000 iterationer
# Hvert skud simuleres som Bernoulli(p_i)
simulated_goals_silkeborg = []

for i in range(1000):
    random_numbers = np.random.rand(len(silkeborg_df))
    goals = random_numbers < silkeborg_df["xg"].values
    total_goals = goals.sum()
    simulated_goals_silkeborg.append(total_goals)

In [ ]:
# Simulation for AaB — 1000 iterationer
# Hvert skud simuleres som Bernoulli(p_i)
simulated_goals_aab = []

for i in range(1000):
    random_numbers = np.random.rand(len(aab_df))
    goals = random_numbers < aab_df["xg"].values
    total_goals = goals.sum()
    simulated_goals_aab.append(total_goals)

In [ ]:
# Gennemsnitlige simulerede mål — AaB
print(f"AaB simulerede mål (gns): {np.mean(simulated_goals_aab):.2f}")
print(f"AaB faktiske mål: {int(aab_df['is_goal'].sum())}")

In [ ]:
# Gennemsnitlige simulerede mål — Silkeborg
print(f"Silkeborg simulerede mål (gns): {np.mean(simulated_goals_silkeborg):.2f}")
print(f"Silkeborg faktiske mål: {int(silkeborg_df['is_goal'].sum())}")

In [ ]:

from collections import Counter

bins = range(0, 7)

sil_counts = Counter(simulated_goals_silkeborg)
aab_counts = Counter(simulated_goals_aab)

sil_probs = [sil_counts.get(g, 0) / 1000 for g in bins]
aab_probs = [aab_counts.get(g, 0) / 1000 for g in bins]

x = np.array(list(bins))
width = 0.4

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, sil_probs, width=width, color="red", alpha=0.8, label="Silkeborg IF")
ax.bar(x + width/2, aab_probs, width=width, color="steelblue", alpha=0.8, label="AaB")

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y*100:.0f}%"))
ax.set_xlabel("Goals")
ax.set_ylabel("Probability")
ax.set_title("Probability of scoring exactly x goals — Silkeborg IF vs AaB")
ax.set_xticks(list(bins))
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Print sandsynligheder for hvert antal mål
print("=== Silkeborg IF ===")
for g in range(7):
    print(f"  P(score exactly {g} goals) = {sil_probs[g]*100:.1f}%")

print("\n=== AaB ===")
for g in range(7):
    print(f"  P(score exactly {g} goals) = {aab_probs[g]*100:.1f}%")
